# Week 1 — Task 4: Pick the Right Chart, Five Times

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

**Dataset:** `mpg.csv` — 398 cars, 1970–1982. `horsepower` is missing for 6 cars.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('data/mpg.csv')
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')
df['origin_name'] = df['origin'].map({1: 'American', 2: 'European', 3: 'Japanese'})
df['year'] = df['model_year'] + 1900

print(f'Shape: {df.shape}')
print(f'Missing horsepower: {df["horsepower"].isna().sum()} cars — excluded from horsepower charts')
df.head()

## Question 1 — Did Cars Become More Fuel-Efficient 1970–1982?

In [ ]:
year_stats = df.groupby('year')['mpg'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(year_stats['year'],
                year_stats['mean'] - year_stats['std'],
                year_stats['mean'] + year_stats['std'],
                alpha=0.15, color='#2166ac')
ax.plot(year_stats['year'], year_stats['mean'], lw=2.5, color='#2166ac', marker='o', ms=5)
ax.set_title('Fleet fuel efficiency rose steadily, accelerating after the 1973 oil shock', loc='left')
ax.set_xlabel('Year')
ax.set_ylabel('Miles per gallon')
ax.set_ylim(0, 40)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task4_q1_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart type: Line chart. Justification: MPG measured over continuous time; a line shows trend direction and rate of change better than any other type.')

## Question 2 — Do American, European and Japanese Cars Differ in Fuel Efficiency?

In [ ]:
origin_order = ['American', 'European', 'Japanese']
origin_means = df.groupby('origin_name')['mpg'].mean().reindex(origin_order)

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#d73027', '#4575b4', '#1a9850']
bars = ax.bar(origin_order, origin_means.values, color=colors, alpha=0.85, edgecolor='white')

for bar, val in zip(bars, origin_means.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f} mpg', ha='center', fontsize=9)

ax.set_title('Japanese cars average 9 mpg more than American cars', loc='left')
ax.set_ylabel('Average miles per gallon')
ax.set_ylim(0, 35)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task4_q2_origin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart type: Bar chart. Justification: comparing a single quantity across three discrete, unordered groups. Bars make the magnitude difference easy to read.')

## Question 3 — What is the Relationship Between Weight and Fuel Efficiency?

In [ ]:
r = df['weight'].corr(df['mpg'])
m, b = np.polyfit(df['weight'], df['mpg'], 1)
xs = np.linspace(df['weight'].min(), df['weight'].max(), 100)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df['weight'], df['mpg'], s=20, alpha=0.4, color='#4575b4', edgecolors='none')
ax.plot(xs, m * xs + b, color='#d73027', lw=2)
ax.set_title(f'Heavier cars are substantially less fuel-efficient (r\u00a0=\u00a0{r:.2f})', loc='left')
ax.set_xlabel('Weight (lbs)')
ax.set_ylabel('Miles per gallon')
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task4_q3_weight.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart type: Scatter plot. Justification: both variables are continuous; scatter reveals relationship direction, strength, and whether it is linear.')

## Question 4 — How is the Number of Cylinders Distributed Across the Fleet?

In [ ]:
cyl_counts = df['cylinders'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(cyl_counts.index.astype(str), cyl_counts.values,
       color='#4575b4', alpha=0.85, edgecolor='white', width=0.6)
ax.set_title('4-cylinder engines dominate; 3 and 5 cylinders are rare one-offs', loc='left')
ax.set_xlabel('Number of cylinders')
ax.set_ylabel('Count of cars')
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task4_q4_cylinders.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart type: Bar chart (discrete distribution). Justification: cylinders is a discrete count variable; a bar chart makes the frequency of each category directly readable. A histogram would imply continuity that does not exist.')

## Question 5 — Did Cars Get Lighter? (Does Weight Explain the MPG Improvement?)

In [ ]:
weight_year = df.groupby('year')['weight'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(weight_year['year'], weight_year['weight'], lw=2.5, color='#d73027', marker='o', ms=5)
ax.set_title('Average weight fell from ~3,400 to ~2,500 lbs — lighter cars partly explain the MPG gain', loc='left')
ax.set_xlabel('Year')
ax.set_ylabel('Average weight (lbs)')
ax.set_ylim(0, 4200)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task4_q5_weight_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart type: Line chart. Justification: same reasoning as Q1 — continuous trend over time.')

## Weakest Chart: Q2 — Redraw as Box Plot

**Why Q2 is the weakest:** The bar chart shows only the mean MPG per origin, hiding all within-group variation. Two origins with identical means could have completely different spreads, outliers, or shapes. A reader who only sees the bars cannot tell whether the difference is driven by a handful of outliers or by a shift in the entire distribution.

**What I changed:** Replaced bar chart (mean only) with box plot (median, IQR, whiskers, outliers). Now the reader sees spread and shape, not just a single summary number.

In [ ]:
data_by_origin = [df[df['origin_name'] == o]['mpg'].values for o in origin_order]

fig, ax = plt.subplots(figsize=(8, 6))
bp = ax.boxplot(data_by_origin, labels=origin_order, patch_artist=True,
                medianprops=dict(color='white', lw=2),
                whiskerprops=dict(lw=1.2),
                capprops=dict(lw=1.2),
                flierprops=dict(marker='o', markerfacecolor='#888888', markersize=4, alpha=0.5))

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_title(
    'Japanese cars have higher median AND less spread than American cars\n'
    '(improved from bar chart: now shows IQR, tails, and outliers)',
    loc='left'
)
ax.set_ylabel('Miles per gallon')
ax.set_ylim(0, 50)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task4_q2_redraw.png', dpi=150, bbox_inches='tight')
plt.show()